In [1]:
!pip install langchain langchain-community langchain-google-genai wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=6bc4d00398c581fea2e1ee23d0196b84c75788bf5e3accd98127f8b67a942510
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-c

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnableLambda

In [ ]:
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Set API key
# os.environ["GOOGLE_API_KEY"] = "your_google_api_key"

# Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Wikipedia tool
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Step 1 — Retrieve context
retriever = RunnableLambda(lambda topic: wiki_tool.run(topic))

# Step 2 — Summarization
summary_prompt = ChatPromptTemplate.from_template(
"""
Summarize the following information in 5 sentences.

Information:
{context}
"""
)

summary_chain = summary_prompt | llm

# Step 3 — Quiz generation
quiz_prompt = ChatPromptTemplate.from_template(
"""
Based on the summary below, generate 3 quiz questions.

Summary:
{summary}
"""
)

quiz_chain = quiz_prompt | llm


In [ ]:

# Full pipeline
pipeline = (
    RunnableMap({"context": retriever})
    | summary_chain
    | RunnableLambda(lambda msg: {"summary": msg.content})
    | quiz_chain
)

# Run pipeline
topic = input("Enter a topic: ")

result = pipeline.invoke(topic)

print("\nQuiz Questions:\n")
print(result.content)